In [8]:
import copy
from heapq import heappush, heappop

# 1. Cấu hình ban đầu
n = 3
# Hướng di chuyển của ô trống: Xuống, Trái, Lên, Phải
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

# 2. Lớp Hàng đợi ưu tiên (Priority Queue)
class priorityQueue:
    def __init__(self):
        self.heap = []
    def push(self, key):
        heappush(self.heap, key)
    def pop(self):
        return heappop(self.heap)
    def empty(self):
        return not self.heap

# 3. Lớp Node đại diện cho mỗi trạng thái của bàn cờ
class nodes:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.costs = costs # Tổng chi phí (f = g + h)
        self.levels = levels # Số bước đi (g)

    # Hàm so sánh để Priority Queue biết node nào có chi phí thấp hơn
    def __lt__(self, nxt):
        return self.costs < nxt.costs

# 4. Các hàm bổ trợ
def calculateCosts(mats, final) -> int:
    """Tính số ô bị sai vị trí (Hamming distance)"""
    count = 0
    for i in range(n):
        for j in range(n):
            # Nếu ô hiện tại khác 0 và khác ô ở đích thì tính là 1 lỗi
            if (mats[i][j]) and (mats[i][j] != final[i][j]):
                count += 1
    return count

def isSafe(x, y):
    """Kiểm tra tọa độ (x, y) có nằm trong bàn cờ không"""
    return 0 <= x < n and 0 <= y < n

def printMatsrix(mats):
    """In ma trận hiện tại ra màn hình"""
    for i in range(n):
        for j in range(n):
            print("%d " % (mats[i][j]), end=" ")
        print()

def printPath(root):
    """In ngược từ kết quả về đầu để thấy các bước di chuyển"""
    if root is None:
        return
    printPath(root.parent)
    printMatsrix(root.mats)
    print()

def newNodes(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> nodes:
    """Tạo một node mới bằng cách di chuyển ô trống"""
    new_mats = copy.deepcopy(mats)

    # Di chuyển ô trống (Swap)
    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    # Chi phí = số bước đi + số ô sai vị trí
    h = calculateCosts(new_mats, final)
    return nodes(parent, new_mats, new_empty_tile_posi, levels + h, levels)

# 5. Hàm giải thuật chính (A*)
def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()

    # Khởi tạo node gốc
    costs = calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_posi, costs, 0)
    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()

        # Nếu chi phí sai lệch bằng số bước (tức h=0), đã tìm thấy đích
        if (minimum.costs - minimum.levels) == 0:
            print("--- Giải thành công! Các bước di chuyển: ---")
            printPath(minimum)
            return

        # Thử di chuyển ô trống theo 4 hướng
        for i in range(4):
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]

            if isSafe(new_tile_posi[0], new_tile_posi[1]):
                child = newNodes(minimum.mats, minimum.empty_tile_posi,
                                 new_tile_posi, minimum.levels + 1,
                                 minimum, final)
                pq.push(child)

# 6. Chạy chương trình
if __name__ == "__main__":
    initial_state = [
        [1, 2, 3],
        [5, 6, 0],
        [7, 8, 4]
    ]

    final_state = [
        [1, 2, 3],
        [5, 8, 6],
        [0, 7, 4]
    ]

    # Tọa độ ô số 0 trong initial_state là dòng 1, cột 2
    empty_pos = [1, 2]

    solve(initial_state, empty_pos, final_state)

--- Giải thành công! Các bước di chuyển: ---
1  2  3  
5  6  0  
7  8  4  

1  2  3  
5  0  6  
7  8  4  

1  2  3  
5  8  6  
7  0  4  

1  2  3  
5  8  6  
0  7  4  



In [9]:
from collections import deque

class Graph:
    def __init__(self, adjac_lis):
        self.adjac_lis = adjac_lis

    def get_neighbors(self, v):
        return self.adjac_lis[v]


    def h(self, n):
        H = {
            'A': 1,
            'B': 1,
            'C': 1,
            'D': 1
        }
        return H[n]

    def a_star_algorithm(self, start, stop):

        open_lst = set([start])
        closed_lst = set([])


        poo = {}
        poo[start] = 0


        par = {}
        par[start] = start

        while len(open_lst) > 0:
            n = None


            for v in open_lst:
                if n == None or poo[v] + self.h(v) < poo[n] + self.h(n):
                    n = v

            if n == None:
                print('Path does not exist!')
                return None


            if n == stop:
                reconst_path = []

                while par[n] != n:
                    reconst_path.append(n)
                    n = par[n]

                reconst_path.append(start)
                reconst_path.reverse()

                print('Path found: {}'.format(reconst_path))
                return reconst_path


            for (m, weight) in self.get_neighbors(n):

                if m not in open_lst and m not in closed_lst:
                    open_lst.add(m)
                    par[m] = n
                    poo[m] = poo[n] + weight


                else:
                    if poo[m] > poo[n] + weight:
                        poo[m] = poo[n] + weight
                        par[m] = n


                        if m in closed_lst:
                            closed_lst.remove(m)
                            open_lst.add(m)


            open_lst.remove(n)
            closed_lst.add(n)

        print('Path does not exist!')
        return None

=

adjac_lis = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)]
}

graph1 = Graph(adjac_lis)
graph1.a_star_algorithm('A', 'C')

Path found: ['A', 'B', 'C']


['A', 'B', 'C']

In [11]:
#BaiTapTaiLop_1
import copy
from heapq import heappush, heappop

# 1. Cấu hình cho 15-puzzle
n = 4
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class PriorityQueue:
    def __init__(self):
        self.heap = []
    def push(self, key):
        heappush(self.heap, key)
    def pop(self):
        return heappop(self.heap)
    def empty(self):
        return not self.heap

class Node:
    def __init__(self, parent, mat, empty_tile_pos, cost, level):
        self.parent = parent
        self.mat = mat
        self.empty_tile_pos = empty_tile_pos
        self.cost = cost   # f(n) = g(n) + h(n)
        self.level = level # g(n)
    def __lt__(self, nxt):
        return self.cost < nxt.cost

# 2. Hàm tính khoảng cách Manhattan (Heuristic h)
def calculateManhattan(mat, final) -> int:
    """
    Tính tổng khoảng cách di chuyển cần thiết của mỗi ô về vị trí đúng.
    Công thức: |x1 - x2| + |y1 - y2|
    """
    dist = 0
    # Tạo bản đồ vị trí đích để tra cứu nhanh
    pos_final = {}
    for i in range(n):
        for j in range(n):
            pos_final[final[i][j]] = (i, j)

    for i in range(n):
        for j in range(n):
            tile = mat[i][j]
            if tile != 0: # Không tính ô trống
                target_i, target_j = pos_final[tile]
                dist += abs(i - target_i) + abs(j - target_j)
    return dist

# 3. Các hàm hỗ trợ tương tự bài 8-puzzle
def isSafe(x, y):
    return 0 <= x < n and 0 <= y < n

def printMatrix(mat):
    for row in mat:
        print(" ".join(f"{val:2d}" for val in row))
    print()

def printPath(root):
    if root is None: return
    printPath(root.parent)
    print(f"Bước {root.level}:")
    printMatrix(root.mat)

def solve(initial, empty_tile_pos, final):
    pq = PriorityQueue()

    # h(n) ban đầu
    h = calculateManhattan(initial, final)
    root = Node(None, initial, empty_tile_pos, h, 0)
    pq.push(root)

    # Để tránh lặp lại các trạng thái đã duyệt (giúp AKT chạy nhanh hơn)
    visited = set()

    while not pq.empty():
        minimum = pq.pop()

        # Chuyển ma trận thành tuple để lưu vào set (vì list không hash được)
        mat_tuple = tuple(tuple(row) for row in minimum.mat)
        if mat_tuple in visited:
            continue
        visited.add(mat_tuple)

        # Kiểm tra nếu đã tới đích (h = 0)
        if minimum.cost - minimum.level == 0:
            print("--- Đã tìm thấy lời giải! ---")
            printPath(minimum)
            return

        # Sinh các node con
        for i in range(4):
            new_x = minimum.empty_tile_pos[0] + rows[i]
            new_y = minimum.empty_tile_pos[1] + cols[i]

            if isSafe(new_x, new_y):
                # Tạo bản sao và swap
                new_mat = copy.deepcopy(minimum.mat)
                x, y = minimum.empty_tile_pos
                new_mat[x][y], new_mat[new_x][new_y] = new_mat[new_x][new_y], new_mat[x][y]

                # Tính f(n) = g(n) + h(n)
                new_level = minimum.level + 1
                new_h = calculateManhattan(new_mat, final)

                child = Node(minimum, new_mat, (new_x, new_y), new_level + new_h, new_level)
                pq.push(child)

# --- Chạy thử ---
if __name__ == "__main__":
    # 0 đại diện cho ô trống
    initial_state = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 0, 11],
        [13, 14, 15, 12]
    ]

    final_state = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 0]
    ]

    # Vị trí số 0 ban đầu là (2, 2)
    start_empty_pos = (2, 2)

    solve(initial_state, start_empty_pos, final_state)

--- Đã tìm thấy lời giải! ---
Bước 0:
 1  2  3  4
 5  6  7  8
 9 10  0 11
13 14 15 12

Bước 1:
 1  2  3  4
 5  6  7  8
 9 10 11  0
13 14 15 12

Bước 2:
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0



In [12]:
#BaiTapTaiLop_2
import heapq

class Graph:
    def __init__(self, adjacency_list, heuristics):
        """
        adjacency_list: Từ điển chứa {đỉnh: [(hàng xóm, trọng số), ...]}
        heuristics: Từ điển chứa {đỉnh: giá trị h(n)} - ước lượng đến đích
        """
        self.adjacency_list = adjacency_list
        self.heuristics = heuristics

    def get_neighbors(self, v):
        return self.adjacency_list.get(v, [])

    def h(self, n):
        # Trả về giá trị Heuristic của đỉnh n
        return self.heuristics.get(n, 0)

    def a_star_algorithm(self, start, stop):
        # open_list: Hàng đợi ưu tiên chứa các node (f_score, node_name)
        # Sử dụng heapq để luôn lấy ra node có f thấp nhất một cách hiệu quả
        open_list = []
        heapq.heappush(open_list, (self.h(start), start))

        # g_score: Chi phí thực tế từ điểm bắt đầu đến node hiện tại
        g_score = {start: 0}

        # parents: Lưu vết đường đi để tái tạo lại kết quả
        parents = {start: start}

        while open_list:
            # Lấy node có f(n) nhỏ nhất
            current_f, n = heapq.heappop(open_list)

            # Nếu tìm thấy đích
            if n == stop:
                path = []
                while parents[n] != n:
                    path.append(n)
                    n = parents[n]
                path.append(start)
                path.reverse()

                print(f"--- Tìm thấy đường đi: {' -> '.join(path)} ---")
                print(f"Tổng chi phí thực tế (g): {g_score[stop]}")
                return path

            # Xét các đỉnh kề (m) và trọng số (weight) của cạnh (n, m)
            for (m, weight) in self.get_neighbors(n):
                # Tính chi phí g tạm thời nếu đi qua n
                tentative_g = g_score[n] + weight

                # Nếu m chưa được duyệt HOẶC tìm thấy đường đi tới m ngắn hơn
                if m not in g_score or tentative_g < g_score[m]:
                    g_score[m] = tentative_g
                    f_score = tentative_g + self.h(m)
                    parents[m] = n
                    heapq.heappush(open_list, (f_score, m))

        print("Không tìm thấy đường đi!")
        return None

# --- Thiết lập dữ liệu đồ thị mẫu ---

# 1. Danh sách kề: { Đỉnh: [(Đỉnh kề, Trọng số cạnh), ...] }
adjac_lis = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 2)],
    'D': [('E', 3)],
    'E': []
}

# 2. Giá trị Heuristic h(n): Khoảng cách ước lượng từ đỉnh n đến đích 'E'
# (Trong thực tế, đây có thể là khoảng cách đường chim bay)
heuristics = {
    'A': 10,
    'B': 8,
    'C': 5,
    'D': 2,
    'E': 0
}

# --- Chạy thử ---
graph = Graph(adjac_lis, heuristics)
path = graph.a_star_algorithm('A', 'E')

--- Tìm thấy đường đi: A -> C -> D -> E ---
Tổng chi phí thực tế (g): 8
